# Atividade: Identificação NARX com Redes Neurais Artificiais (PyTorch)

Neste notebook, vamos adaptar o benchmark de Narendra para utilizar os dados reais do **Aeropêndulo**. O objetivo é treinar uma Rede Neural Artificial para modelar o sistema e avaliar o desempenho em predição One-Step-Ahead (OSA) e Free-Run (Simulação Livre).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# set random seed (reproducibility)
rseed = 42
np.random.seed(rseed)
torch.manual_seed(rseed)

# print precision
np.set_printoptions(precision=3)

## 1. Funções Auxiliares

Vamos definir a função `matReg` para montar as matrizes de regressão e a função `freeRun` para simulação livre. Utilizaremos as matrizes adaptadas que geram o regressor no formato adequado para o PyTorch.

In [ ]:
def matReg(y, u, ny, nu):
    p = max(ny, nu) + 1
    (N, ) = y.shape
    (Nu, ) = u.shape

    # sanity check
    if N != Nu:
        print('Dimensions of u and y vector are not consistent')
        return (-1,-1)

    # create target vector
    target = y[p-1:N]

    # create regression matrix
    Phi = np.zeros((N-p+1, ny+nu))
    for i in range(ny):
        Phi[:, i] = y[p-i-2 : N-i-1]

    for i in range(nu):
        Phi[:, i+ny] = u[p-i-2 : N-i-1]

    return (target, Phi)


def freeRun(model, y, u, ny, nu):
    p = max(ny, nu) + 1
    (N, ) = y.shape

    yhat = np.zeros(N)
    yhat[:p-1] = y[:p-1] # include initial conditions

    for k in range(p, N+1):
        auxY = np.concatenate((yhat[(k-p):(k-1)], (0,)), axis=0)
        auxU = np.concatenate((u[(k-p):(k-1)], (0,)), axis=0)

        _, fr_input = matReg(auxY, auxU, ny, nu)
        
        # Converter regressor para tensor e fazer predição
        tensor_input = torch.tensor(fr_input, dtype=torch.float32)
        yhat[k-1] = model.predict(tensor_input).item()
        
    return yhat[-(N-p+1):]

## 2. Carregamento e Preparação dos Dados (Aeropêndulo)

Em vez de dados sintéticos, carregamos o caso de estudo com as respostas ao degrau.

In [ ]:
def carregar_experimento(url, decimacao=1):
    df = pd.read_csv(url)
    df_sub = df.iloc[::decimacao].copy().reset_index(drop=True)
    u_raw = df_sub['motor_percent'].values
    y_raw = df_sub['angulo_deg'].values + 90.0
    # Remove offset
    u = u_raw - u_raw[0]
    y = y_raw  - y_raw[0]
    return u, y

BASE = (
    "https://raw.githubusercontent.com/FelipeEduardoMarcondes/"
    "SYSTEM-IDENTIFICATION-AERO/main/PYTHON/dados/"
)

ue, ye = carregar_experimento(BASE + "step_35_open.csv") # Estimação
ut, yt = carregar_experimento(BASE + "step_39_open.csv") # Teste (Validação)

plt.figure(figsize=(12, 8))
plt.subplot(221)
plt.plot(ue, color='green')
plt.title('ue (Entrada Estimação)')
plt.grid()
plt.subplot(222)
plt.plot(ut, color='orange')
plt.title('ut (Entrada Teste)')
plt.grid()
plt.subplot(223)
plt.plot(ye, color='blue')
plt.title('ye (Saída Estimação)')
plt.grid()
plt.subplot(224)
plt.plot(yt, color='red')
plt.title('yt (Saída Teste)')
plt.grid()
plt.tight_layout()
plt.show()

## 3. Matrizes de Regressão e Tensores PyTorch

Construímos as matrizes usando as ordens do modelo. Escolhemos valores consistentes (como $ny=2, nu=2$, ou maiores dependendo do seu projeto).

In [ ]:
ny = 2
nu = 2 # model orders

(Ye, Phie) = matReg(ye, ue, ny, nu)
(Yt, Phit) = matReg(yt, ut, ny, nu)

# Converter para tensores do PyTorch
Phie_t = torch.tensor(Phie, dtype=torch.float32)
Ye_t   = torch.tensor(Ye, dtype=torch.float32).view(-1, 1)

Phit_t = torch.tensor(Phit, dtype=torch.float32)
Yt_t   = torch.tensor(Yt, dtype=torch.float32).view(-1, 1)

# DataLoader de treino
dataset = TensorDataset(Phie_t, Ye_t)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# DataLoader de teste
dataset_test = TensorDataset(Phit_t, Yt_t)
dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

## 4. Definição da Arquitetura da Rede Neural (PyTorch)

Usaremos uma rede Perceptron Multicamadas (MLP) com três camadas ocultas (função de ativação SELU).

In [ ]:
class MyModel(nn.Module):
    def __init__(self, ninp, nneu, nout):
        super(MyModel, self).__init__()
        self.hidden1 = nn.Linear(ninp, nneu)
        self.hidden2 = nn.Linear(nneu, nneu)
        self.hidden3 = nn.Linear(nneu, nneu)
        self.output = nn.Linear(nneu, nout)

        # Initialize weights with LeCun Normal Initialization
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity='selu')
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity='selu')
        nn.init.kaiming_normal_(self.hidden3.weight, nonlinearity='selu')
        nn.init.kaiming_normal_(self.output.weight, nonlinearity='linear')

    def forward(self, x):
        x = F.selu(self.hidden1(x))
        x = F.selu(self.hidden2(x))
        x = F.selu(self.hidden3(x))
        x = self.output(x)
        return x

    def predict(self, x):
        with torch.no_grad():
            return self.forward(x)

_, ninp = Phie.shape
nout = 1
nneu = 40  # Neurônios por camada (ajustável)

model = MyModel(ninp, nneu, nout)
optimizer = optim.NAdam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

## 5. Treinamento do Modelo

Vamos realizar o loop de treinamento.

In [ ]:
epochs = 30  # Aumentamos o número de épocas para os dados reais convergirem melhor
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(dataloader):
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(dataloader)
    
    # Printar a cada 5 épocas
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')

print("Treinamento finalizado!")

## 6. One-Step-Ahead (OSA) Prediction

In [ ]:
# One step ahead
y_train_pred1 = model.predict(Phie_t).numpy()
y_test_pred1  = model.predict(Phit_t).numpy()

R2test1  = r2_score(Yt, y_test_pred1)
R2train1 = r2_score(Ye, y_train_pred1)

print('-------------- One step ahead  --------------')
print(f'R2 Train: {R2train1:.4f}')
print(f'R2 Test : {R2test1:.4f}')

## 7. Free-Run Simulation

Simulação recursiva onde a rede usa suas próprias predições passadas como entrada.

In [ ]:
# Free Run apenas nos dados de teste
print("Executando Free-Run...")
y_test_pred0  = freeRun(model, yt, ut, ny, nu)

R2test0  = r2_score(Yt, y_test_pred0)

print('-------------- Free Run  --------------')
print(f'R2 Test (FR): {R2test0:.4f}')

## 8. Gráficos de Resultados

In [ ]:
t_test = np.arange(len(Yt))

plt.figure(figsize=(14, 5))
plt.plot(t_test, Yt, 'k', lw=2, label='Real (Teste)')
plt.plot(t_test, y_test_pred0, 'r', ls='--', lw=1.5, label='Prediction (FR)')
plt.title('Free-Run Simulation: Test Dataset')
plt.xlabel('Amostras (k)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(6,6))
minY = min(Yt.min(), y_test_pred0.min())
maxY = max(Yt.max(), y_test_pred0.max())
plt.scatter(Yt, y_test_pred0, c='red', label='Prediction', alpha=0.5)
plt.plot([minY, maxY], [minY, maxY], color='black', linewidth=2, label='Perfect model')
plt.xlabel('Real')
plt.ylabel('Prediction')
plt.title('Scatter Plot - Free-Run Validation')
plt.grid()
plt.legend()
plt.show()